# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
취소·환불 주문을 제외한 **완료(completed) 주문 기준으로**, 어떤 카테고리·상품·월·고객에서 매출이 만들어지는가?

### 사용한 파일/컬럼
- `order_items.csv`: `order_id`, `product_id`, `quantity`, `unit_price` (→ `line_total` 계산)
- `orders.csv`: `order_id`, `customer_id`, `order_date`, `order_status`
- `products.csv`: `product_id`, `product_name`, `category`
- `customers.csv`: `customer_id`, `city` (이름 `name`은 사용하지 않음)

### 결과 관찰
- 파일 크기: customers 150행, products 100행, orders 300행, order_items 764행.
- 주문 상태: completed 184건, cancelled 64건, refunded 52건.
- 전체 주문 상세 금액은 255,610,000원이고, completed 기준 매출은 148,990,000원(상세 474행, 184개 주문)입니다.

### 나의 해석과 판단
금액은 `order_items`에 있지만 주문 상태는 `orders`에만 있고, 카테고리는 `products`에만 있습니다. 그래서 세 테이블을 연결해야 질문에 답할 수 있다고 판단했습니다. 개인정보인 고객 이름은 분석에 필요하지 않아 제외했고, 고객은 `customer_id`와 익명 라벨(`Customer 117`)로만 표현했습니다. 범위를 completed로 한정한 이유는 취소·환불 주문이 포함되면 매출이 과대 집계되기 때문입니다.

### 업무·분석적 의미
같은 데이터라도 범위를 어떻게 잡느냐에 따라 매출 규모가 달라집니다(전체 2.56억 vs completed 1.49억). 보고서에는 항상 "어떤 범위의 매출인지"를 명시해야 합니다.

### 한계와 추가 확인 사항
- 환불 금액·부분 환불·할인·배송비·쿠폰은 데이터에 없어서 반영하지 못했습니다.
- `order_status`에는 3가지 값만 있어 보류·배송 중 같은 상태는 구분할 수 없습니다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: `order_status == 'completed'` (고객 탐색 단계에서는 `age >= 30`, `city == '서울'`, `city.isin(['서울','부산'])`도 사용)
- 정렬 기준: 상품 가격 내림차순, 고객 나이 내림차순, 카테고리·상품·고객은 `total_sales` 내림차순, 월별은 `order_month` 오름차순
- 만든 파생 컬럼: `line_total`, `order_month`(`order_date`를 월 단위 문자열로 변환), `sales_ratio`, `average_order_value`
- `line_total` 계산식: `line_total = quantity * unit_price`

![필터와 파생 컬럼](images/step02_transform.png)
<!-- 캡처 위치: 노트북 6번(line_total 생성)·8번(completed 필터) 셀 Output -->

### 결과 관찰
- 30세 이상 고객은 111명, 서울 고객 15명, 서울·부산 고객 31명입니다.
- 가장 비싼 상품은 `뷰티 상품 099`(200,000원)입니다.
- 날짜 변환 실패는 0건이고, 전체 764행 중 completed는 474행입니다.

### 나의 해석과 판단
필터 기준이 달라지면 결과가 크게 달라집니다. 이번 데이터에서 취소·환불까지 포함하면 255,610,000원(cancelled 62,181,000원, refunded 44,439,000원 포함)이고, completed만 쓰면 148,990,000원으로 약 41.7% 줄어듭니다. 도시 필터도 `'서울'` 대신 `'Seoul'`로 쓰면 0건이 나옵니다. 그래서 필터 전에 `value_counts()`로 실제 값을 확인하는 것이 중요합니다.

### 업무·분석적 의미
"매출"이라는 말이 주문 전체인지 확정 매출인지를 먼저 합의해야 같은 지표를 두고 팀 간 숫자가 어긋나는 일을 막을 수 있습니다.

### 한계와 추가 확인 사항
`refunded`가 전액 환불인지 부분 환불인지 알 수 없습니다. 또한 `age >= 30` 같은 조건은 경계값(30세 포함 여부) 정의를 확인해야 합니다.

## 3. merge 검증
- 병합한 데이터: ① `order_items` + `orders` (주문 상태·고객·날짜 연결), ② completed 주문 상세 + `products` (카테고리·상품명 연결)
- 사용한 key: ① `order_id` ② `product_id`
- `validate` 결과: 두 병합 모두 `many_to_one` 통과(오류 없음). `orders.order_id` 중복 0건, `products.product_id` 중복 0건
- `indicator` 결과: ① both 764 / left_only 0 / right_only 0, ② both 474 / left_only 0 / right_only 0
- 병합 전/후 행 수: ① 764 → 764, ② 474 → 474

![merge 검증](images/step03_merge.png)
<!-- 캡처 위치: 노트북 7번(주문 병합)·9번(상품 병합) 셀 Output -->

### 결과 관찰
두 병합 모두 행 수가 그대로이고, 모든 행이 양쪽 테이블에서 매칭(both)되었으며, 중복 키도 없었습니다.

### 나의 해석과 판단
이번 병합은 안전하다고 판단합니다. 근거는 세 가지입니다. (1) 오른쪽 테이블의 key가 고유해서 `validate='many_to_one'`을 통과했고, (2) `left_only`가 0이라 주문 상태나 카테고리가 비는 행이 없으며, (3) 병합 전후 행 수가 같아 행이 복제되지 않았습니다. 행 수가 늘거나 줄지 않았으므로 별도로 설명할 변화는 없습니다. 추가로 `unit_price`가 `products.price`와 모두 일치하는지도 확인했고(불일치 0건), 주문 상세의 `product_id` 중 상품 테이블에 없는 값도 0건이었습니다.

### 업무·분석적 의미
오른쪽 테이블 key가 중복된 상태로 병합하면 행이 복제되어 매출이 부풀려지고, 반대로 key가 매칭되지 않으면 카테고리나 상태가 비어 집계에서 누락됩니다. 둘 다 오류 메시지 없이 숫자만 틀어지므로 눈에 잘 띄지 않습니다.

### 한계와 추가 확인 사항
`validate`와 `indicator`는 키 구조와 매칭 여부만 확인합니다. 키는 맞지만 값이 잘못 입력된 경우(예: 엉뚱한 고객 ID)는 잡지 못합니다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의: `order_status == 'completed'`인 주문의 주문 상세(474행, 184개 주문, 2025-10-08 ~ 2026-10-04). 매출 = `quantity * unit_price`의 합. 할인·환불·배송비 미반영.
- 카테고리별 결과: 스포츠 31,743,000원(21.31%), 전자기기 26,400,000원(17.72%), 생활용품 23,915,000원(16.05%), 뷰티 23,383,000원(15.69%), 식품 16,573,000원(11.12%), 도서 16,389,000원(11.00%), 패션 10,587,000원(7.11%)
- 상품별 결과: 1위 `스포츠 상품 041` 5,705,000원(35개), 2위 `식품 상품 012` 4,375,000원(25개), 3위 `스포츠 상품 009` 3,860,000원(20개)
- 월별 결과: 2026-01이 25,766,000원(26건)으로 최고, 2026-10은 923,000원(1건)으로 최저
- 고객별 결과: 매출 1위 `Customer 117`(성남) 4,100,000원(5건), 2위 `Customer 102`(고양) 3,996,000원, 3위 `Customer 83`(수원) 3,880,000원. 완료 주문 고객은 100명.

![핵심 집계 결과](images/step04_groupby.png)
<!-- 캡처 위치: 노트북 12번(카테고리·상품), 13번(월별), 14번(고객) 셀 Output -->

### 결과 관찰
- 카테고리: 스포츠가 1위이고 패션이 최하위입니다. 상위 2개(스포츠·전자기기)가 전체의 약 39%를 차지합니다.
- 수량과 매출 순위가 다릅니다. 도서는 수량 149개로 식품(133개)보다 많지만 매출은 식품이 더 큽니다.
- 상품: `스포츠 상품 041` 하나가 5,705,000원으로 2위 상품보다 약 30% 많습니다.
- 월별: 2026-01에 급증(25,766,000원)했다가 2026-02에 8,812,000원으로 줄었습니다. 이후 2026-04(16,981,000원)와 2026-07(17,553,000원)에 다시 높아졌습니다.
- 월별 평균 주문 금액은 647,632원(2026-05) ~ 991,000원(2026-01) 범위입니다.
- 고객: 매출 1위 고객이 전체의 약 2.75%, 상위 10명이 약 23.3%를 차지합니다.

### 나의 해석과 판단
가장 중요한 결과는 **카테고리 간 매출 격차와 2026-01의 월별 급증**이라고 판단했습니다. 카테고리는 상품 구성·재고 의사결정과 바로 연결되고, 2026-01은 주문 수(26건)와 평균 주문 금액(991,000원)이 모두 높아 원인을 확인할 가치가 있기 때문입니다. 반면 고객 1명의 비중이 3% 미만이라 특정 고객 의존도는 낮다고 봅니다.

### 업무·분석적 의미
스포츠·전자기기 중심의 재고·마케팅 우선순위를 검토하고, 2026-01 급증이 프로모션·계절성·특정 대형 주문 중 무엇 때문인지 분해한 뒤, 매출은 높지만 수량이 적은 카테고리(식품·패션 등)의 가격·구성을 별도로 점검할 수 있습니다.

### 한계와 추가 확인 사항
`total_sales`는 `quantity * unit_price`를 합한 값일 뿐이어서 회계상 순매출과 같다고 단정할 수 없습니다. 이유는 다음과 같습니다.
- 할인·쿠폰·포인트 사용액이 반영되지 않았습니다.
- 배송비·세금(부가세) 처리 여부를 알 수 없습니다.
- completed 이후 발생하는 환불·반품이 `order_status` 한 값으로만 표현되어 부분 환불을 알 수 없습니다.
- 매출 인식 시점(주문일 vs 결제일 vs 배송 완료일)이 다를 수 있고, 여기서는 `order_date`를 기준으로 했습니다.
- 마지막 달(2026-10)은 데이터가 10월 초까지만 있어 부분 월입니다. 월별 비교 시 완결된 달과 같은 기준으로 해석하면 안 됩니다.

## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148,990,000
- 카테고리 합계: 148,990,000 (31,743,000 + 26,400,000 + 23,915,000 + 23,383,000 + 16,573,000 + 16,389,000 + 10,587,000)
- 월별 합계: 148,990,000 (13개월 합산)
- 고객별 합계: 148,990,000 (완료 주문 고객 100명 합산, 데이터 파일로 별도 계산)
- 차이 여부: 모두 0원 차이 (불일치 없음)

![총합 검증](images/step05_total_check.png)
<!-- 캡처 필요: 노트북에는 총합 비교 셀이 없으므로 아래 코드를 새 셀로 실행해 Output을 캡처하세요 -->

```python
base_total = completed_order_sales['line_total'].sum()
checks = {
    '원본 completed': base_total,
    '카테고리 합계': category_sales['total_sales'].sum(),
    '월별 합계': monthly_summary['total_sales'].sum(),
    '고객별 합계': customer_sales['total_sales'].sum(),
}
for name, value in checks.items():
    print(f'{name}: {value:,} / 차이: {value - base_total:,}')
```

### 나의 해석과 판단
집계표는 groupby, merge, 필터를 거치면서 만들어지기 때문에 중간에 행이 복제되거나 누락되어도 각 표만 보면 이상이 드러나지 않습니다. 모든 집계의 총합이 원본 합계와 일치하는지 확인하면 이런 오류를 잡을 수 있습니다. 불일치가 나오면 다음 순서로 확인합니다. (1) 병합 전후 행 수와 `validate`·`indicator` 결과, (2) key 중복 여부, (3) 필터 조건이 표마다 동일한지, (4) 결측값(`NaN`)이 groupby에서 빠졌는지(`dropna`), (5) 날짜 변환 실패(`NaT`)로 월별에서 빠진 행이 있는지.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약: LLM이 제안한 `category_sales = order_items.groupby('category')['line_total'].sum()` 코드가 현재 데이터에서 바로 실행 가능한지 검토하고, category 컬럼 위치, 주문 상태 필터링, merge `validate`·`indicator`를 포함한 안전한 수정 코드와 검증 순서를 요청
- 제안 코드 요약: `order_items`에서 바로 `category`로 groupby하여 `line_total`을 합산
- 실제 컬럼/범위와 맞지 않은 부분:
  1. `order_items`에는 `category` 컬럼이 없습니다(`category`는 `products.csv`에 있음). 그대로 실행하면 `KeyError: 'category'`가 납니다.
  2. 이 시점의 `order_items`는 주문 상태 정보가 없어 취소·환불 주문이 포함됩니다(전체 255,610,000원).
  3. 병합 검증(`validate`, `indicator`)이 없어 행 복제나 누락을 확인할 수 없습니다.
- 수정한 내용: `order_items`와 `orders`를 `order_id`로 병합(`many_to_one`, `indicator=True`)한 뒤 `completed`만 필터링하고, `products`를 `product_id`로 병합(`many_to_one`, `indicator=True`)하여 `category`를 붙인 뒤 groupby. 마지막에 총합 비교로 검증(노트북 7~12번 셀).
- 최종 판단: **수정 후 사용**

![LLM 코드 검증](images/step06_llm_validation.png)
<!-- 캡처 위치: 14번 LLM 프롬프트 셀 + 수정 코드(7~12번) 실행 결과 -->

### 나의 해석과 판단
코드가 에러 없이 실행된다는 것은 문법과 컬럼명이 맞다는 뜻일 뿐, 분석 질문에 맞는 결과라는 보장은 아닙니다. 예를 들어 카테고리를 먼저 붙인 뒤 상태 필터를 빼면 코드는 정상 실행되지만 결과는 취소·환불 포함 금액(255,610,000원)이 됩니다. 병합 키가 중복되면 에러 없이 매출이 부풀려질 수도 있습니다. 그래서 실제 컬럼명, 필터 범위, 병합 검증, 총합 일치를 사람이 직접 확인해야 합니다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 분석 범위 선택이 매출 규모를 좌우한다. 전체 주문 상세 금액은 255,610,000원이지만 completed 기준은 148,990,000원으로, 취소·환불 106,620,000원(약 41.7%)이 제외된다.
2. completed 기준 카테고리 매출은 스포츠(21.31%)와 전자기기(17.72%)가 상위이고, 월별로는 2026-01(25,766,000원)이 가장 높은 피크이다.

### 그 결과를 뒷받침하는 수치/표
- 범위별 금액: 전체 255,610,000원 = completed 148,990,000원 + cancelled 62,181,000원 + refunded 44,439,000원
- 카테고리 상위: 스포츠 31,743,000원, 전자기기 26,400,000원, 생활용품 23,915,000원
- 월별 최고: 2026-01 25,766,000원, 26건, 평균 주문 금액 991,000원
- 총합 검증: 카테고리·월별·고객별 합계 모두 148,990,000원으로 원본과 일치

### 추가로 확인하고 싶은 질문
- 2026-01 급증은 프로모션, 계절성, 일부 대형 주문 중 무엇 때문인가?
- 취소·환불 비율이 높은 카테고리나 결제수단이 있는가? (결제수단별 주문 수: 카카오페이 79, 네이버페이 77, 계좌이체 74, 카드 70으로 비슷함)
- 수량 대비 매출이 높은 상품은 가격 정책을 조정할 여지가 있는가?

### 현재 결과의 한계
- 샘플 데이터이므로 실제 비즈니스 결론으로 일반화할 수 없다.
- 할인·배송비·세금·부분 환불이 없어 `total_sales`는 회계상 순매출이 아니다.
- 2026-10은 부분 월이라 월별 비교에 주의가 필요하다.
- 카테고리별 비중은 한 해 정도의 기간만 본 것이어서 추세를 단정하기 어렵다.

## 최종 제출 체크
- [ ] 핵심 셀 Output이 남아 있습니다.
- [ ] merge와 총합 검증 Evidence가 있습니다.
- [ ] 결과 관찰과 해석이 구분되어 있습니다.
- [ ] LLM 코드를 검증했습니다.
- [ ] 개인정보/Secret이 없습니다.
- [ ] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.